# Phase3: 拡張あり vs 拡張なし

> ⏱ 所要時間: 説明を読む 8分 ＋ 実行 約3分（`QUICK = True` のとき）

## このNotebookのゴール
**同じ条件で「拡張あり」と「拡張なし」を比べ、データ拡張が過学習を抑えて、初めて見る写真への正解率を上げるかを確かめる。**
さらに、**拡張には「学習が遅くなる」という副作用がある**ことを理解する。

## 前のNotebookとのつながり
Notebook 02 で、拡張なしの成績（ベースライン）と過学習を確認しました。
今回は同じAI・同じ学習回数で、**拡張の有無だけ**を変えます。

> **比較の鉄則: 条件を1つだけ変える。**
> AIの形・学習率・エポック数・シードは同じにします。変えるのは「拡張の有無」だけ。
> そうしないと、成績の差が「拡張のせい」なのか「他の違いのせい」なのか分かりません。

## このNotebookでやること
1. 学習中、写真が**どう加工されて**AIに渡されているかを見る
2. 拡張なし vs 拡張あり（`basic`）で学習して比べる
3. 学習曲線と、成績の表を読む
4. 学習回数（エポック）によって結論が変わることを知る

## 使う拡張 `basic`
**ランダム切り抜き → 左右反転 → 色調変更**の3つの組み合わせ（Notebook 01 で見たものです）。

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

## 設定
`QUICK = True` は各条件1回、`False` は3回（平均を取る本番版）。`EPOCHS = ep(150)` は**150周**学習します。
（150周にする理由は、このNotebookの最後の考察で分かります）

In [ ]:
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = ep(150)

## 1. 学習中、写真はどう見えている？

学習のしくみは次のとおりです。

- **エポック（周回）ごとに**、同じ1枚の写真に拡張を**やり直す**。だから毎回違う見え方でAIに渡される。
- 拡張は **train だけ**。val・test は拡張せず、**元のまま**（リサイズと正規化のみ）評価する。

次のセルは、**同じ1枚の写真が、8エポックぶんでどう見えるか**を表示します。

In [ ]:
from torchvision import datasets

# train用のデータセット。写真を読むたびに basic 拡張がかかる
ds = datasets.ImageFolder(DATA / "train", T.make_transform("basic"))

# 同じ1枚（ds[0]）を8回取り出す＝8エポックぶん。毎回違う加工がかかっている
shown = [ds[0][0] for _ in range(8)]
show_images(shown, [f"エポック{i+1}" for i in range(8)], cols=4);

**見るポイント**: 同じアリなのに、**位置・大きさ・向き・色**が毎回違います。
AIは「こういう見え方のアリ」「あんな見え方のアリ」と、**1枚の写真から何通りも**学べます。
色合いが毎回違うのは、拡張の「色調変更」がかかっているためです。

## 2. 拡張なし vs 拡張あり（basic）

次のセルは、`none`（拡張なし）と `basic`（拡張あり）を、**同じ条件**で学習して比べます。
**時間がかかります**（`QUICK = True` で約3分）。実行中は `[*]` と表示されます。完了まで待ってください。

In [ ]:
res = {}
for aug in ["none", "basic"]:
    print(f"== aug={aug} ==")
    res[aug] = run_seeds(SEEDS, aug=aug, epochs=EPOCHS)   # (結果の表, 各回の詳細) を保存

df = pd.concat([d for d, _ in res.values()])
mean_table(df)    # 条件ごとの平均 ± ばらつき

**表の見方**
- `aug` 列: `none` が拡張なし、`basic` が拡張あり。
- `train_acc`: 学習用の写真の正解率。 `val_acc` / `test_acc`: 初めて見る写真の正解率（**こちらが本当の実力**）。
- `gap`: train − val。小さいほど過学習が少ない。

次に、学習曲線を**上下に並べて**比べます（上: 拡張なし、下: 拡張あり）。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for row, aug in enumerate(["none", "basic"]):
    h = res[aug][1][0]["hist"]            # 最初のシードの学習履歴
    x = [e["epoch"] for e in h]
    axes[row, 0].plot(x, [e["train_loss"] for e in h], label="train")
    axes[row, 0].plot(x, [e["val_loss"] for e in h], label="val")
    axes[row, 0].set_title(f"{aug}: loss（小さいほど良い）")
    axes[row, 1].plot(x, [e["train_acc"] for e in h], label="train")
    axes[row, 1].plot(x, [e["val_acc"] for e in h], label="val")
    axes[row, 1].set_title(f"{aug}: 精度（大きいほど良い）")
    for a in axes[row]:
        a.legend(); a.grid(alpha=0.3)
fig.tight_layout();

**見るポイント**
- **上段（拡張なし）**: train loss は下がるのに、val loss は下がらず（上がることも）、train と val の精度が**大きく離れている**。→ 過学習。
- **下段（拡張あり）**: train と val の差が**小さい**。train loss の下がり方も**ゆるやか**（AIが丸暗記しにくい）。

## 3. 結果の読み方（MANUAL.mdの記載値・3シード平均）

学習回数（エポック）を60回と150回にしたときの比較です。

| エポック | 条件 | train | val | test | gap |
|---|---|---|---|---|---|
| 60 | 拡張なし | 0.859 | 0.719 | 0.809 | 0.139 |
| 60 | 拡張あり | 0.756 | 0.715 | 0.757 | 0.041 |
| 150 | 拡張なし | 0.945 | 0.711 | 0.701 | 0.235 |
| 150 | 拡張あり | 0.822 | 0.732 | 0.796 | 0.089 |

### 何が分かるか

1. **60エポックでは、拡張の効果が見えない。**
   gap は小さくなったが、test は拡張なし（0.809）のほうがむしろ高い。**拡張ありは、学習が遅い**（train 精度が 0.756 までしか上がっていない）。

2. **150エポックまで学習すると、差が出る。**
   拡張なしは、学習を続けるほど過学習が進み、**test が 0.809 → 0.701 に悪化**する。
   拡張ありは過学習せず、**test が 0.796 を維持**する。→ 拡張の効果が現れた（約 +9.5 ポイント）。

3. **教訓: 拡張は「過学習を抑える」代わりに「学習が遅くなる」。**
   → **十分な回数、学習してから比べないと、拡張の効果を見誤る。**
   （60エポックだけで比べたら、「拡張しても効果なし」と間違った結論を出していた）

> 🍎 たとえ話: 毎回問題を少し変えて出す先生（拡張あり）のクラスは、丸暗記ができない分、**身に付くのに時間がかかる**。でも、時間をかければ**初見の問題に強く**なる。

### 注意: 数字の信頼性
val・test は約77枚しかなく、結果に**ばらつき**があります（標準偏差0.02〜0.04程度）。
**0.02〜0.04程度の差は誤差の可能性が高い**ので、過信しないでください。

## 🧪 試してみよう

### 演習1: 学習回数を短くして、結論が変わるのを見る
**やること**: 設定のセルの **`EPOCHS = ep(150)`** を **`EPOCHS = ep(60)`** に書き換え、設定のセル以降を実行し直します（所要時間は半分以下）。
- **見るポイント**: 拡張あり（basic）の train 精度は、拡張なしより**低くなっていませんか？** test の差は、150エポックのときより小さい／逆転していませんか？
- **分かること**: **学習が足りないと、拡張の効果が見えない**。「拡張は効かない」と早合点しないための練習です。

### 演習2: 3回の平均で確かめる
**やること**: 設定のセルの **`QUICK = True`** を **`False`** にして、設定のセル以降を実行し直します（約8分）。
- **見るポイント**: 表の `±`（ばらつき）。差が `±` より大きいなら、**意味のある差**の可能性が高い。
- **分かること**: 1回の結果だけでは、差が本物か偶然か分からない。

### 演習3: 切り抜きの強さを変える
**やること**: `src/train.py` を開き、`AUG_SETS` の `"basic"` の中の **`scale=(0.6, 1.0)`** を **`scale=(0.3, 1.0)`** に書き換えて保存します。
その後、**カーネルを再起動**（メニューの「カーネルを再起動」）して、このNotebookを上から実行し直します。
- `scale` は「元の写真の何割を切り出すか」。`(0.3, 1.0)` は、より大きく切り取る（強い拡張）。
- **見るポイント**: 拡張が強くなると、学習曲線や test 精度はどう変わりますか？
- ⚠ 終わったら、`(0.6, 1.0)` に**戻してください**。

### 考えてみよう（コード不要）
拡張なしのほうが **train 精度は高い**のに、**test 精度は低い**のはなぜでしょう？
- ヒント: train の精度が高い＝丸暗記が進んでいる。test が低い＝初めて見る写真に弱い。

---

## ✅ まとめ

- 拡張で、train と val の差（**過学習**）が縮む。十分に学習すれば、test の成績も上がる。
- ただし、拡張には**学習が遅くなる**副作用がある。**学習回数が少ないと、効果が見えない。**
- 比較は**条件を1つだけ変える**。結果は**複数シードの平均**で見る。差が誤差範囲でないか、`±` で確認する。

➡ 次は **`04_ablation.ipynb`**。拡張を**1種類ずつ**試して**どの拡張が効いているのか**を調べます。さらに、写真の枚数を変えたときの効果も見ます。